<div style="padding:28px 32px;border:1px solid #d0d7de;border-radius:14px;">
  <div style="font-size:14px;letter-spacing:.08em;text-transform:uppercase;"><b>Complete Study Guide</b></div>
  <h1 style="margin:.35em 0 .15em 0;">Every Theory Behind the Options Pricing Calculator</h1>
  <p style="font-size:18px;margin:.2em 0 0 0;">
    Payoffs, arbitrage, probability, Black-Scholes-Merton, Greeks, implied volatility,
    binomial trees, Monte Carlo simulation, and the exact Python lines that implement them.
  </p>
</div>

<br>

This notebook explains the complete project rather than a simplified toy version.

> **Educational warning:** A model price is conditional on assumptions. It is not a guaranteed fair value, executable quote, or trading profit.

## How to use this notebook

- Read the Markdown sections for finance and mathematical theory.
- Run the code demonstrations to see the formulas numerically.
- Place this notebook inside the project folder to run cells importing the real modules.

## Contents

### Part I — Option foundations
1. What derivatives are
2. Calls and puts
3. Long and short positions
4. Strike, expiry, premium, and multiplier
5. Moneyness
6. Intrinsic value and time value
7. Payoff versus profit
8. European and American exercise

### Part II — No-arbitrage foundations
9. Time value of money
10. Continuous compounding
11. Dividend yield and forwards
12. No-arbitrage bounds
13. Put-call parity
14. Replication
15. Risk-neutral valuation

### Part III — Black-Scholes-Merton
16. Random returns and Brownian motion
17. Geometric Brownian motion
18. Lognormal prices
19. Model assumptions
20. The Black-Scholes formulas
21. The meaning of d1 and d2
22. Limiting cases and validation

### Part IV — Greeks
23. Delta
24. Gamma
25. Vega
26. Theta
27. Rho
28. Units and interpretation
29. Delta hedging
30. Taylor approximation

### Part V — Implied volatility
31. Historical volatility
32. Implied volatility
33. Root finding
34. Brent's method
35. Solver bounds and failures
36. Smiles, skews, and surfaces

### Part VI — Binomial trees
37. One-step model
38. Up and down factors
39. Risk-neutral probability
40. Recombining trees
41. Backward induction
42. American exercise
43. Convergence

### Part VII — Monte Carlo
44. Random sampling
45. Risk-neutral paths
46. Discounted expected payoff
47. Standard error
48. Confidence intervals
49. Antithetic variates
50. Control variates
51. Convergence and use cases

### Part VIII — The actual project
52. Project architecture
53. Input validation
54. Black-Scholes code map
55. Greek code map
56. Implied-volatility code map
57. Binomial code map
58. Monte Carlo code map
59. Price surfaces
60. Greek profiles
61. Market option chains
62. Automated tests
63. Limitations and improvements
64. Formula sheet
65. Glossary
66. Revision questions and answers

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

pd.set_option("display.float_format", lambda value: f"{value:,.6f}")
print("Notebook setup complete.")

# Part I — Option foundations

---

# 1. What derivatives are

A derivative is a contract whose value depends on another asset or variable.

For an equity option, the underlying is normally a share or ETF. The option's value depends on the underlying price, strike, time, volatility, interest rates, dividends, and exercise rules.

# 2. Calls and puts

A **call** gives its holder the right to buy the underlying at the strike price.

A **put** gives its holder the right to sell the underlying at the strike price.

Symbols used throughout the project:

- $S$ — current spot price
- $K$ — strike price
- $T$ — time to expiry in years
- $r$ — continuously compounded risk-free rate
- $q$ — continuous dividend yield
- $\sigma$ — annualized volatility
- $C$ — call value
- $P$ — put value

# 3. Long and short positions

The long option buyer pays a premium and owns the right.

The short option writer receives the premium and has the corresponding obligation.

At expiry, the writer's payoff is the negative of the holder's payoff, before transaction fees and credit effects.

# 4. Strike, expiry, premium, and multiplier

- **Strike**: contractual exercise price.
- **Expiry**: final date on which contractual rights exist.
- **Premium**: price paid for the option.
- **Multiplier**: number of underlying units represented by one contract.

The project's pricing functions return value per one underlying unit.

If the multiplier is 100, a quoted option value of 4.25 corresponds to:

$$
4.25 \times 100 = 425
$$

# 5. Moneyness

For a call:

- In the money when $S > K$
- At the money when $S \approx K$
- Out of the money when $S < K$

For a put, the direction is reversed.

Moneyness describes the relationship between spot and strike. It does not by itself determine whether a trade is profitable because premium matters.

# 6. Intrinsic value and time value

Call intrinsic value:

$$
\max(S-K,0)
$$

Put intrinsic value:

$$
\max(K-S,0)
$$

Time value is the amount of premium above intrinsic value:

$$
\text{Time value}
=
\text{Option premium}
-
\text{Intrinsic value}
$$

In [ ]:
spot = 110
strike = 100
call_intrinsic = max(spot - strike, 0)
put_intrinsic = max(strike - spot, 0)

pd.Series({
    "call_intrinsic": call_intrinsic,
    "put_intrinsic": put_intrinsic,
})

# 7. Payoff versus profit

Payoff ignores the premium paid.

Profit includes the premium.

Long call payoff:

$$
\max(S_T-K,0)
$$

Long call profit:

$$
\max(S_T-K,0)-C_0
$$

Long put profit:

$$
\max(K-S_T,0)-P_0
$$

In [ ]:
terminal_spots = np.linspace(50, 150, 201)
strike = 100
premium = 8

call_payoff = np.maximum(terminal_spots - strike, 0)
call_profit = call_payoff - premium

plt.figure(figsize=(9, 4))
plt.plot(terminal_spots, call_payoff, label="Call payoff")
plt.plot(terminal_spots, call_profit, label="Call profit")
plt.axhline(0, linewidth=1)
plt.axvline(strike, linestyle="--")
plt.xlabel("Terminal spot")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()

# 8. European and American exercise

A European option can be exercised only at expiry.

An American option can be exercised at any time up to expiry.

Black-Scholes-Merton directly prices European contracts. The project's binomial tree handles both European and American exercise.

# Part II — No-arbitrage foundations

---

# 9. Time value of money

A future payment is worth less than the same amount today because today's money can earn interest.

The present value of strike $K$ paid at time $T$ is:

$$
K e^{-rT}
$$

# 10. Continuous compounding

Future value:

$$
FV=PV e^{rT}
$$

Present value:

$$
PV=FV e^{-rT}
$$

The standard Black-Scholes-Merton notation uses continuously compounded rates.

In [ ]:
present_value = 100
rate = 0.05
years = 2
future_value = present_value * np.exp(rate * years)
recovered_present_value = future_value * np.exp(-rate * years)

future_value, recovered_present_value

# 11. Dividend yield and forwards

With continuous dividend yield $q$, the theoretical forward price is:

$$
F_0=S_0e^{(r-q)T}
$$

Interest increases carrying value. Dividends reduce the forward because the holder of the physical asset receives distributions before expiry.

# 12. No-arbitrage bounds

European call bounds:

$$
\max\left(Se^{-qT}-Ke^{-rT},0\right)
\leq C \leq
Se^{-qT}
$$

European put bounds:

$$
\max\left(Ke^{-rT}-Se^{-qT},0\right)
\leq P \leq
Ke^{-rT}
$$

The implied-volatility solver rejects prices outside these bounds.

# 13. Put-call parity

For European options with the same underlying, strike, and expiry:

$$
C-P
=
Se^{-qT}
-
Ke^{-rT}
$$

Equivalent portfolio identity:

$$
C+Ke^{-rT}
=
P+Se^{-qT}
$$

Both sides create the same expiry payoff, so no-arbitrage requires equal present values.

# 14. Replication

In a one-period model, create a portfolio containing $\Delta$ units of underlying and a bond position $B$.

The portfolio must satisfy:

$$
\Delta S_u + Be^{rT}=V_u
$$

$$
\Delta S_d + Be^{rT}=V_d
$$

Subtracting the equations gives:

$$
\Delta=
\frac{V_u-V_d}{S_u-S_d}
$$

# 15. Risk-neutral valuation

A perfectly replicated derivative cannot earn an arbitrary risk premium without creating arbitrage.

The pricing identity is:

$$
V_0
=
e^{-rT}
\mathbb{E}^{\mathbb{Q}}
\left[
\text{Payoff at }T
\right]
$$

$\mathbb{Q}$ is the risk-neutral pricing measure. It is a valuation device, not a claim that real investors are indifferent to risk.

# Part III — Black-Scholes-Merton

---

# 16. Random returns and Brownian motion

Brownian motion $W_t$ is a continuous stochastic process with independent normally distributed increments.

Over a small interval $dt$:

$$
dW_t \sim N(0,dt)
$$

It supplies the random shock in geometric Brownian motion.

# 17. Geometric Brownian motion

Under the real-world measure:

$$
dS_t
=
(\mu-q)S_t\,dt
+
\sigma S_t\,dW_t
$$

Under risk-neutral pricing, $\mu$ is replaced by $r$:

$$
dS_t
=
(r-q)S_t\,dt
+
\sigma S_t\,dW_t^{\mathbb{Q}}
$$

# 18. Lognormal prices

The risk-neutral terminal price is:

$$
S_T
=
S_0
\exp\left(
\left(r-q-\frac{1}{2}\sigma^2\right)T
+
\sigma\sqrt{T}Z
\right)
$$

where:

$$
Z\sim N(0,1)
$$

The logarithm of price is normal, so price is lognormal and remains positive.

# 19. Model assumptions

Black-Scholes-Merton assumes:

1. No arbitrage
2. Frictionless trading
3. Continuous trading and hedging
4. Constant volatility
5. Constant risk-free rate
6. Continuous dividend yield
7. Lognormal underlying prices
8. No jumps
9. European exercise
10. Idealized borrowing and short selling

# 20. The Black-Scholes formulas

Define:

$$
d_1
=
\frac{
\ln(S/K)
+
\left(r-q+\frac{1}{2}\sigma^2\right)T
}{
\sigma\sqrt{T}
}
$$

$$
d_2
=
d_1-\sigma\sqrt{T}
$$

Call:

$$
C
=
Se^{-qT}N(d_1)
-
Ke^{-rT}N(d_2)
$$

Put:

$$
P
=
Ke^{-rT}N(-d_2)
-
Se^{-qT}N(-d_1)
$$

In [ ]:
def black_scholes_call(S, K, T, r, sigma, q=0.0):
    d1 = (
        np.log(S / K)
        + (r - q + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    return (
        S * np.exp(-q * T) * norm.cdf(d1)
        - K * np.exp(-r * T) * norm.cdf(d2)
    )

black_scholes_call(100, 100, 1, 0.05, 0.20)

# 21. The meaning of $d_1$ and $d_2$

$d_1$ and $d_2$ standardize the exercise threshold under the model's lognormal distribution.

- $N(d_1)$ appears in call delta.
- $N(d_2)$ is connected to the risk-neutral probability that a call finishes in the money.

They are not directly real-world forecasting probabilities.

# 22. Limiting cases and validation

At expiry:

$$
C_T=\max(S_T-K,0)
$$

$$
P_T=\max(K-S_T,0)
$$

At zero volatility, uncertainty disappears and value becomes the discounted deterministic payoff.

The project explicitly handles expiry and zero volatility instead of dividing by zero in $d_1$.

# Part IV — Greeks

---

# 23. Delta

Delta is first-order sensitivity to spot:

$$
\Delta=\frac{\partial V}{\partial S}
$$

Call delta:

$$
\Delta_C=e^{-qT}N(d_1)
$$

Put delta:

$$
\Delta_P=e^{-qT}\left(N(d_1)-1\right)
$$

# 24. Gamma

Gamma measures how delta changes with spot:

$$
\Gamma=\frac{\partial^2V}{\partial S^2}
$$

For a European call or put:

$$
\Gamma
=
\frac{e^{-qT}\phi(d_1)}{S\sigma\sqrt{T}}
$$

Gamma is normally largest around the strike and close to expiry.

# 25. Vega

Vega measures sensitivity to volatility:

$$
\text{Vega}=\frac{\partial V}{\partial \sigma}
$$

$$
\text{Vega}
=
Se^{-qT}\phi(d_1)\sqrt{T}
$$

The raw derivative is for a volatility change of 1.00. The project divides by 100 and reports value change per one volatility percentage point.

# 26. Theta

Theta measures sensitivity to the passage of time:

$$
\Theta=\frac{\partial V}{\partial t}
$$

The project returns annual theta and approximate daily theta:

$$
\Theta_{day}=\frac{\Theta_{year}}{365}
$$

Theta is local and changes as spot, volatility, and time change.

# 27. Rho

Rho measures sensitivity to the risk-free rate:

$$
\rho=\frac{\partial V}{\partial r}
$$

Call rho:

$$
\rho_C=KTe^{-rT}N(d_2)
$$

Put rho:

$$
\rho_P=-KTe^{-rT}N(-d_2)
$$

The project reports rho per one interest-rate percentage point.

# 28. Units and interpretation

- Delta: option-value change per one unit of spot
- Gamma: delta change per one unit of spot
- Vega: option-value change per one volatility point in this project
- Theta: option-value change per year or calendar day
- Rho: option-value change per one rate point in this project

Always state units before comparing Greek values.

# 29. Delta hedging

A long option with delta $\Delta$ can be made approximately spot-neutral by holding:

$$
-\Delta
$$

units of underlying per option unit.

The hedge is only local because delta changes when spot, time, and volatility move.

# 30. Taylor approximation

A local change in option value can be approximated by:

$$
\Delta V
\approx
\Delta\,\Delta S
+
\frac{1}{2}\Gamma(\Delta S)^2
+
\text{Vega}\,\Delta\sigma
+
\Theta\,\Delta t
+
\rho\,\Delta r
$$

The approximation deteriorates for large simultaneous changes.

# Part V — Implied volatility

---

# 31. Historical volatility

Historical volatility estimates realized movement from past returns.

A common estimator is:

$$
\widehat{\sigma}_{annual}
=
s_{daily}\sqrt{252}
$$

The market-data module estimates it using close-to-close log returns.

# 32. Implied volatility

Implied volatility is the volatility input that matches an observed option price.

It solves:

$$
V_{BSM}(\sigma)-V_{market}=0
$$

It is model-implied, not directly observed physical volatility.

# 33. Root finding

The project defines a pricing error function:

```python
return black_scholes_price(trial) - market_price
```

The implied volatility is the root where the error equals zero.

# 34. Brent's method

The project calls SciPy's `brentq` with a lower and upper volatility.

The method combines reliable bracketing with fast interpolation steps.

A valid bracket requires the pricing error to have opposite signs at the interval endpoints.

# 35. Solver bounds and failures

The solver can fail when:

- The option price violates no-arbitrage bounds
- Time to expiry is zero
- The market quote is stale or crossed
- The volatility interval does not bracket a root
- Model assumptions are incompatible with the contract

The project validates bounds before solving.

# 36. Smiles, skews, and surfaces

If the model were exact, contracts sharing an underlying and expiry would have one volatility.

Real implied volatility varies across strike and expiry, creating:

- Smiles
- Skews
- Term structures
- Volatility surfaces

This is evidence that constant volatility and lognormal returns are incomplete descriptions of real markets.

# Part VI — Binomial trees

---

# 37. One-step model

In one time step, the underlying moves from $S$ to either:

$$
S_u=Su
$$

or:

$$
S_d=Sd
$$

The derivative is valued by replication or equivalent risk-neutral expectation.

# 38. Up and down factors

The Cox-Ross-Rubinstein tree uses:

$$
u=e^{\sigma\sqrt{\Delta t}}
$$

$$
d=\frac{1}{u}
$$

The reciprocal construction makes the tree recombine.

# 39. Risk-neutral probability

The pricing probability is:

$$
p
=
\frac{e^{(r-q)\Delta t}-d}{u-d}
$$

It is chosen so the expected underlying growth matches risk-free carry under the pricing measure.

It is not a forecast of the real probability of an up move.

# 40. Recombining trees

An up move followed by a down move gives:

$$
Sud=S
$$

A down move followed by an up move also gives:

$$
Sdu=S
$$

Therefore the number of unique nodes grows quadratically rather than exponentially.

# 41. Backward induction

At expiry, each node equals intrinsic value.

Move backward using:

$$
V
=
e^{-r\Delta t}
\left(pV_u+(1-p)V_d\right)
$$

Repeating this calculation eventually produces the root price.

# 42. American exercise

At every American node:

$$
V_{American}
=
\max\left(
V_{continuation},
V_{exercise}
\right)
$$

A European node uses continuation only.

This local maximum is the main reason trees are useful for American options.

# 43. Convergence

As step count increases, a European CRR value generally converges toward Black-Scholes-Merton under matching assumptions.

Convergence can oscillate, so the error does not necessarily decrease monotonically at every step count.

# Part VII — Monte Carlo

---

# 44. Random sampling

Monte Carlo draws standard normal shocks:

$$
Z_i\sim N(0,1)
$$

Each draw creates one possible terminal price and payoff.

# 45. Risk-neutral paths

The project simulates terminal prices directly:

$$
S_T^{(i)}
=
S_0\exp\left(
\left(r-q-\frac{1}{2}\sigma^2\right)T
+
\sigma\sqrt{T}Z_i
\right)
$$

Because the option is European, intermediate path points are unnecessary.

# 46. Discounted expected payoff

Call estimator:

$$
\widehat{C}
=
e^{-rT}\frac{1}{N}
\sum_{i=1}^{N}
\max\left(S_T^{(i)}-K,0\right)
$$

Put estimator:

$$
\widehat{P}
=
e^{-rT}\frac{1}{N}
\sum_{i=1}^{N}
\max\left(K-S_T^{(i)},0\right)
$$

# 47. Standard error

If discounted simulated payoffs have sample standard deviation $s$:

$$
SE=\frac{s}{\sqrt{N}}
$$

Monte Carlo convergence is slow:

$$
SE=O\left(N^{-1/2}\right)
$$

Halving standard error requires roughly four times as many paths.

# 48. Confidence intervals

An approximate confidence interval is:

$$
\widehat{V}
\pm
z_{1-\alpha/2}SE
$$

For 95% confidence:

$$
z_{0.975}\approx1.96
$$

The interval measures simulation uncertainty conditional on the model, not total model risk.

# 49. Antithetic variates

For every shock $Z$, also simulate $-Z$.

The two paths are negatively related, which can reduce estimator variance without changing the expected value.

# 50. Control variates

The discounted terminal underlying has known expectation:

$$
\mathbb{E}^{\mathbb{Q}}\left[e^{-rT}S_T\right]
=
S_0e^{-qT}
$$

The project uses the difference between simulated and known underlying expectation to reduce payoff noise.

# 51. Convergence and use cases

For a plain European option, Black-Scholes-Merton is faster and exact under its assumptions.

Monte Carlo becomes valuable for:

- Path-dependent contracts
- Multiple risk factors
- Complex payoffs
- High-dimensional problems
- Models without closed-form solutions

# Part VIII — The actual project

---

# 52. Project architecture

| File | Responsibility |
|---|---|
| `black_scholes.py` | Inputs, prices, Greeks, bounds, parity |
| `implied_volatility.py` | Volatility inversion |
| `binomial.py` | European and American CRR trees |
| `monte_carlo.py` | Risk-neutral simulation and variance reduction |
| `analytics.py` | Comparisons, surfaces, Greek profiles |
| `market_data.py` | Optional price history and option chains |
| `main.py` | Complete export workflow |
| `app.py` | Interactive Streamlit calculator |
| `tests/` | Mathematical and software validation |

# 53. Input validation

The `OptionInputs.validate()` method rejects:

- Non-positive spot
- Non-positive strike
- Negative time
- Negative volatility
- Invalid option types
- Rates or dividend yields at or below -100%

Defensive checks prevent formulas from silently producing meaningless numbers.

# 54. Black-Scholes code map

Core line:

```python
price = discounted_spot * norm.cdf(d1) - discounted_strike * norm.cdf(d2)
```

Theory mapping:

```text
discounted_spot   → S exp(-qT)
discounted_strike → K exp(-rT)
norm.cdf(d1)      → N(d1)
norm.cdf(d2)      → N(d2)
```

# 55. Greek code map

Example gamma line:

```python
gamma = discount_spot * density / (
    spot * volatility * sqrt_time
)
```

This implements:

$$
\Gamma
=
\frac{e^{-qT}\phi(d_1)}{S\sigma\sqrt{T}}
$$

# 56. Implied-volatility code map

The solver creates new `OptionInputs` values while changing only volatility.

```python
return black_scholes_price(trial) - market_price
```

Then:

```python
brentq(pricing_error, lower_volatility, upper_volatility)
```

The root is the implied volatility.

# 57. Binomial code map

Continuation value:

```python
continuation = discount * (
    probability * value_up
    + (1 - probability) * value_down
)
```

American node:

```python
values = np.maximum(continuation, exercise)
```

# 58. Monte Carlo code map

Terminal price:

```python
terminal_spots = spot * np.exp(drift + diffusion)
```

Discounted payoff:

```python
discounted_payoffs = np.exp(-r * T) * payoffs
```

Estimator:

```python
price = np.mean(adjusted_payoffs)
```

# 59. Price surfaces

The project recalculates Black-Scholes-Merton value over grids of spot and volatility.

A surface helps visualize:

- Moneyness
- Convexity
- Vega
- Interaction between spot and volatility

# 60. Greek profiles

The project holds strike, time, rates, dividends, and volatility fixed while changing spot.

This shows that Greeks are state-dependent functions, not permanent contract constants.

# 61. Market option chains

The optional `market_data.py` module obtains listed expirations and chains.

When bid and ask are both positive, midpoint is:

$$
\text{Midpoint}
=
\frac{\text{Bid}+\text{Ask}}{2}
$$

Market data can be delayed, stale, missing, crossed, or unsuitable for execution.

# 62. Automated tests

The test suite checks:

- Known call and put benchmark prices
- Put-call parity
- No-arbitrage bounds
- Delta ranges
- Binomial convergence
- American value not below European value
- Non-dividend American call equality
- Recovery of known implied volatility
- Monte Carlo confidence coverage
- Random-seed reproducibility
- Invalid-input rejection

# 63. Limitations and improvements

Important limitations:

1. Constant volatility
2. Constant interest rate
3. Continuous dividend yield
4. No jumps
5. Lognormal prices
6. Frictionless trading
7. Continuous hedging
8. No liquidity or transaction costs
9. Simplified American exercise model
10. Unit price excludes contract multiplier
11. Market quotes may be stale
12. Greeks are local approximations
13. Confidence intervals exclude model uncertainty
14. Historical and implied volatility are different

Professional extensions include local volatility, stochastic volatility, jump diffusion, calibrated volatility surfaces, finite-difference PDE solvers, and portfolio risk aggregation.

# 64. Formula sheet

## Call payoff

$$
\max(S_T-K,0)
$$

## Put payoff

$$
\max(K-S_T,0)
$$

## Forward price

$$
F_0=S_0e^{(r-q)T}
$$

## Put-call parity

$$
C-P=Se^{-qT}-Ke^{-rT}
$$

## d1

$$
d_1
=
\frac{\ln(S/K)+\left(r-q+\frac{1}{2}\sigma^2\right)T}{\sigma\sqrt{T}}
$$

## d2

$$
d_2=d_1-\sigma\sqrt{T}
$$

## Call price

$$
C=Se^{-qT}N(d_1)-Ke^{-rT}N(d_2)
$$

## Put price

$$
P=Ke^{-rT}N(-d_2)-Se^{-qT}N(-d_1)
$$

## Binomial probability

$$
p=\frac{e^{(r-q)\Delta t}-d}{u-d}
$$

## Monte Carlo standard error

$$
SE=\frac{s}{\sqrt{N}}
$$

# 65. Glossary

| Term | Meaning |
|---|---|
| American option | Exercisable before or at expiry |
| Arbitrage | Riskless profit from inconsistent prices under assumptions |
| Call | Right to buy |
| Delta | First derivative with respect to spot |
| Gamma | Change in delta with spot |
| Implied volatility | Volatility input matching market price |
| Intrinsic value | Immediate-exercise value |
| Monte Carlo | Simulation-based numerical valuation |
| Premium | Option price |
| Put | Right to sell |
| Risk-neutral measure | Pricing measure using risk-free expected growth |
| Rho | Sensitivity to interest rate |
| Strike | Exercise price |
| Theta | Sensitivity to time |
| Vega | Sensitivity to volatility |

# 66. Revision questions

1. What right does a call provide?
2. What is the difference between payoff and profit?
3. Why is strike discounted?
4. State put-call parity.
5. What does risk-neutral valuation mean?
6. Why are terminal prices lognormal in the model?
7. State the principal Black-Scholes assumptions.
8. What does call delta measure?
9. Why does gamma create rehedging risk?
10. In what units does this project report vega?
11. Why must implied volatility be solved numerically?
12. Why must market price satisfy bounds?
13. What is the CRR up factor?
14. What does risk-neutral tree probability represent?
15. How is an American node valued?
16. Why can an American put exceed a European put?
17. How is Monte Carlo price estimated?
18. How does standard error scale with path count?
19. What do antithetic variates do?
20. What is the project's control variate?

# Answers

1. The right to buy the underlying at the strike.
2. Profit includes premium and other cash costs; payoff does not.
3. The strike is paid in the future and must be converted to present value.
4. $C-P=Se^{-qT}-Ke^{-rT}$.
5. Discount expected payoff under the pricing measure with risk-free carry.
6. Geometric Brownian motion makes log price normal.
7. Constant volatility and rates, lognormal prices, frictionless continuous trading, no arbitrage, and European exercise, among others.
8. First-order sensitivity of call value to spot.
9. Delta changes when spot changes.
10. Per one volatility percentage point.
11. Volatility cannot be isolated with an elementary closed-form inverse.
12. Otherwise no bracketed model-consistent solution exists and arbitrage restrictions are violated.
13. $u=e^{\sigma\sqrt{\Delta t}}$.
14. A pricing weight that produces risk-free expected carry.
15. Maximum of continuation value and immediate exercise value.
16. Receiving the strike earlier can be valuable.
17. Discount the average simulated payoff.
18. As $1/\sqrt{N}$.
19. Pair $Z$ with $-Z$ to reduce variance.
20. Discounted terminal underlying value with known expectation $S_0e^{-qT}$.

<div style="padding:20px 24px;border:1px solid #d0d7de;border-radius:12px;">
<h2 style="margin-top:0;">Final mental model</h2>

```text
Contract payoff
    ↓
No-arbitrage and replication
    ↓
Risk-neutral valuation
    ↓
Analytical formula, tree, or simulation
    ↓
Greeks, implied volatility, and scenarios
    ↓
Validation against identities and numerical convergence
```

A pricing model is a disciplined translation from assumptions to a conditional value.
</div>